### np.fft.fft

In [5]:
import math
import cmath


def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _dft_naive(x):
    """Naive O(N^2) DFT — used as fallback when N is not a power of 2."""
    N = len(x)
    out = []
    for k in range(N):
        val = 0 + 0j
        for n in range(N):
            angle = -2 * math.pi * k * n / N
            val += x[n] * cmath.exp(1j * angle)
        out.append(val)
    return out


def _fft_radix2(x):
    """
    Cooley-Tukey Radix-2 FFT.
    Requires len(x) to be a power of 2.
    """
    N = len(x)

    if N == 1:
        return [complex(x[0])]

    if N == 0:
        return []

    # split into even and odd
    even = _fft_radix2(x[0::2])
    odd  = _fft_radix2(x[1::2])

    T = [cmath.exp(-2j * math.pi * k / N) * odd[k] for k in range(N // 2)]

    out = [even[k] + T[k] for k in range(N // 2)] + \
          [even[k] - T[k] for k in range(N // 2)]
    return out


def _is_power_of_2(n):
    return n > 0 and (n & (n - 1)) == 0


def np_fft_fft(a, n=None, axis=-1, norm=None):
    """
    Pure-Python equivalent of np.fft.fft.
    Computes the 1-D Discrete Fourier Transform.
    Only supports 1-D input for now (axis support is simplified).
    n: if given, truncate or zero-pad input to this length.
    norm: 'backward' (default), 'ortho', or 'forward'.
    Returns: list of complex numbers.
    """
    shape = get_shape(a)
    ndim = len(shape)

    if ndim == 0:
        raise ValueError("Input must be at least 1-D")

    if ndim != 1:
        raise NotImplementedError("Only 1-D input supported in this version")

    x = [complex(v) for v in a]

    # handle n: truncate or zero-pad
    if n is not None:
        if not isinstance(n, int) or n <= 0:
            raise ValueError("n must be a positive integer")
        if n < len(x):
            x = x[:n]
        elif n > len(x):
            x = x + [0j] * (n - len(x))

    N = len(x)

    if N == 0:
        return []

    # choose algorithm
    if _is_power_of_2(N):
        result = _fft_radix2(x)
    else:
        result = _dft_naive(x)

    # apply normalization
    if norm is None or norm == 'backward':
        pass
    elif norm == 'forward':
        result = [v / N for v in result]
    elif norm == 'ortho':
        factor = math.sqrt(N)
        result = [v / factor for v in result]
    else:
        raise ValueError(f"Unknown norm mode: '{norm}'")

    return result

### test cases 

In [6]:
print(np_fft_fft([1, 0, 0, 0]))
print(np_fft_fft([1, 1, 1, 1]))
print(np_fft_fft([1, -1, 1, -1]))
print(np_fft_fft([0, 1, 0, -1]))
print(np_fft_fft([1, 2, 3, 4, 5, 6, 7, 8]))
print(np_fft_fft([1, 2, 3]))
print(np_fft_fft([1, 0, 0, 0], n=8))
print(np_fft_fft([1, 0, 0, 0, 0, 0, 0, 0], norm='ortho'))
print(np_fft_fft([1, 0, 0, 0, 0, 0, 0, 0], norm='forward'))
print(np_fft_fft([]))

[(1+0j), (1+0j), (1+0j), (1+0j)]
[(4+0j), 0j, 0j, 0j]
[0j, 0j, (4+0j), 0j]
[0j, (1.2246467991473532e-16-2j), 0j, (-1.2246467991473532e-16+2j)]
[(36+0j), (-4+9.65685424949238j), (-4+4j), (-4+1.6568542494923797j), (-4+0j), (-4-1.6568542494923806j), (-3.9999999999999996-4j), (-3.9999999999999987-9.65685424949238j)]
[(6+0j), (-1.5000000000000009+0.8660254037844382j), (-1.4999999999999987-0.8660254037844404j)]
[(1+0j), (1+0j), (1+0j), (1+0j), (1+0j), (1+0j), (1+0j), (1+0j)]
[(0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j), (0.35355339059327373+0j)]
[(0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j), (0.125+0j)]
[]


In [7]:
print(np_fft_fft(42)) 

ValueError: Input must be at least 1-D

In [8]:
print(np_fft_fft([[1, 2], [3, 4]])) 

NotImplementedError: Only 1-D input supported in this version

In [9]:
print(np_fft_fft([1, 2, 3], n=0)) 

ValueError: n must be a positive integer

In [10]:
print(np_fft_fft([1, 2, 3], n=-5)) 

ValueError: n must be a positive integer

In [11]:
print(np_fft_fft([1, 2, 3], norm='invalid')) 

ValueError: Unknown norm mode: 'invalid'